# 05 · Quattro modi di far autenticare un device: segreto condiviso, thumbprint, CA propria, CA di ADR

Stesso device, stesso Hub, **quattro credenziali**. Per ognuna generiamo davvero il materiale in locale e guardiamo tre cose:
**cosa conserva il cloud**, **chi emette** e **cosa succede quando scade o trapela**. Poi la tabella di confronto, costruita dai dati.

Le prime quattro parti generano il materiale in locale (chiavi e certificati veri, ma nessuna chiamata ad Azure). Le ultime due lavorano sull'**Hub e su ADR**:
gli stessi casi con due device identity di prova, e la **vista dell'operation**: un inventario solo contro un giro su ogni Hub.
Servono i notebook 01 e 02a-02c già eseguiti.

**Da dire.** "Nella pratica quasi tutti usano il segreto condiviso, o al massimo il thumbprint, perché sono comodi. Oggi voglio
spingere sul certificato completo: vediamo cosa si guadagna, passo per passo, e potete rifarlo a casa."

In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

## 1 · Segreto condiviso (SAS, chiave simmetrica)
Il device e l'Hub conoscono **lo stesso segreto**. Comodo: una stringa. Ma il segreto **esiste in due posti** (device e registry dell'Hub),
non ha scadenza propria e per cambiarlo bisogna riscriverlo su entrambi.

In [ ]:
$sas = @{
    Primary   = [Convert]::ToBase64String([Security.Cryptography.RandomNumberGenerator]::GetBytes(32))
    Secondary = [Convert]::ToBase64String([Security.Cryptography.RandomNumberGenerator]::GetBytes(32))
}
"Chiave primaria (e' un segreto: sta sul device E nel registry dell'Hub): $($sas.Primary.Substring(0,8))........"
"Scadenza della chiave: nessuna. Chi la copia e' il device, ovunque si trovi."

Chiave primaria (e' un segreto: sta sul device E nel registry dell'Hub): G+gRNquV........

Scadenza della chiave: nessuna. Chi la copia e' il device, ovunque si trovi.

## 2 · Certificato self-signed + thumbprint
Il device ha un certificato che si è firmato da solo. L'Hub **non lo riceve**: conserva solo l'**impronta** (thumbprint). Meglio del
segreto, perché la chiave privata **non lascia il device**; ma ogni device è una fiducia a sé, e la rotazione richiede
due impronte (primaria e secondaria) da gestire a mano.

In [ ]:
$dir = Join-Path $out 'sim\03\selfsigned'; New-Item -ItemType Directory -Force $dir | Out-Null
$ecdsa = [Security.Cryptography.ECDsa]::Create([Security.Cryptography.ECCurve]::CreateFromFriendlyName('nistP384'))
$req = [Security.Cryptography.X509Certificates.CertificateRequest]::new('CN=robot-01', $ecdsa, [Security.Cryptography.HashAlgorithmName]::SHA384)
$self = $req.CreateSelfSigned([DateTimeOffset]::UtcNow.AddMinutes(-5), [DateTimeOffset]::UtcNow.AddDays(365))
Write-Pem -Path (Join-Path $dir 'device-key.pem') -Label 'PRIVATE KEY' -Bytes $ecdsa.ExportPkcs8PrivateKey()
Write-Pem -Path (Join-Path $dir 'device-cert.pem') -Label 'CERTIFICATE' -Bytes $self.RawData
[pscustomobject]@{
    "L'Hub conserva"      = "solo il thumbprint: $($self.Thumbprint)"
    'Soggetto = Emittente' = "$($self.Subject) = $($self.Issuer)"
    'Scade'               = $self.NotAfter.ToString('yyyy-MM-dd')
} | Format-List | Out-String


L'Hub conserva       : solo il thumbprint: 588F0B5B16B0B0BD46E35C5D10B561A082487365
Soggetto = Emittente : CN=robot-01 = CN=robot-01
Scade                : 2027-11-16



## 3 · Certificato firmato da una CA tua (DPS classico)
Ora c'è una **catena**: una CA che hai creato tu firma i certificati dei device; DPS (o l'Hub) si fida della **CA**, non di ogni device.
Un nuovo device non richiede nessuna modifica al cloud. Il prezzo: **la chiave della CA è un file sul tuo computer**, e rinnovi e revoche sono compiti tuoi.

In [ ]:
$caDir = Join-Path $out 'sim\03\ca-propria'
$ca = New-LocalCa -Name 'La mia CA di progetto' -Dir $caDir
$leaf = New-DemoLeaf -CaDir $caDir -Cn 'robot-01' -Dir (Join-Path $out 'sim\03\ca-signed') -Days 90
$s = Get-CertSummary -Path $leaf.Chain
[pscustomobject]@{
    'Il cloud si fida di' = 'la CA (caricata e verificata su DPS)'
    'Soggetto'            = $s.Subject
    'Emittente'           = $s.Issuer
    'Scade'               = $s.NotAfter.ToString('yyyy-MM-dd')
    'Chiave della CA'     = "$($ca.Key)  <- sul tuo disco: da proteggere"
} | Format-List | Out-String


Il cloud si fida di : la CA (caricata e verificata su DPS)
Soggetto            : CN=robot-01
Emittente           : CN=La mia CA di progetto
Scade               : 2027-02-14
Chiave della CA     : D:\Dev\2026\IoTHub\WPC2026\artifacts\sim\03\ca-propria\ca-key.pem  <- sul tuo disco: da proteggere



## 4 · Certificato emesso da ADR
La CA (Root + Issuing CA + policy) è **una risorsa del namespace**, con chiavi in un HSM gestito: non c'è nessun file da proteggere.
Il device manda **solo la CSR**; DPS la gira ad ADR; torna una catena. La **durata** la decide la policy.

Se hai eseguito il notebook 02c usiamo il certificato **vero** di `robot-01`; altrimenti una Issuing CA locale al suo posto.

In [ ]:
# Il certificato vero, emesso da ADR (notebook 02c): l'ultima registrazione di robot-01
$real = Get-ChildItem (Join-Path $out 'devices\robot-01') -Directory -ErrorAction SilentlyContinue | Sort-Object Name | Select-Object -Last 1
$chain = if ($real) { Join-Path $real.FullName 'issued-chain.pem' }
if (-not ($chain -and (Test-Path $chain))) { throw "Manca il certificato di robot-01: esegui prima il notebook 02c." }
$s = Get-CertSummary -Path $chain
$n = ([regex]::Matches([IO.File]::ReadAllText($chain), 'BEGIN CERTIFICATE')).Count
[pscustomobject]@{
    'Origine'              = 'emesso da ADR (notebook 02c)'
    'Soggetto'             = "$($s.Subject)   (= registration ID)"
    'Emittente'            = $s.Issuer
    'Scade'                = "$($s.NotAfter.ToString('yyyy-MM-dd')) ($($s.Days) giorni)"
    'Certificati nel file' = $n
    'Chiave privata'       = 'generata sul device, mai inviata'
} | Format-List | Out-String


Origine              : REALE (emesso da ADR, notebook 02c)
Soggetto             : CN=robot-01   (= registration ID)
Emittente            : CN=ica-abb10e496f23bbbf
Scade                : 2026-12-17 (30 giorni)
Certificati nel file : 3
Chiave privata       : generata sul device, mai inviata



## 5 · Il confronto
Costruito dai quattro casi appena visti. Le righe su **rinnovo** e **revoca** sono quelle che cambiano la vita di chi fa operation.

> Nota sulla revoca con ADR: la documentazione Microsoft **non è univoca** (una pagina dice che in preview la revoca non è supportata,
> altre la descrivono; il comando `az iot adr ns device auth revoke-certs` esiste). Il notebook 09 la esegue sul servizio.

In [ ]:
$rows = @(
  [ordered]@{ Aspetto = 'Cosa conserva il cloud';        Segreto = 'il segreto stesso';              Thumbprint = "solo l'impronta";                 CA_propria = 'il certificato della CA';          ADR = 'la CA (nel namespace)' },
  [ordered]@{ Aspetto = 'Chi emette';                    Segreto = 'tu (una stringa)';               Thumbprint = 'il device (self-signed)';         CA_propria = 'la tua CA, un file sul tuo disco'; ADR = 'ADR: Issuing CA con chiavi in HSM' },
  [ordered]@{ Aspetto = 'Se il registry trapela';        Segreto = 'tutti i device sono clonabili';  Thumbprint = 'nulla di utilizzabile';           CA_propria = 'nulla di utilizzabile';            ADR = 'nulla di utilizzabile' },
  [ordered]@{ Aspetto = 'Durata';                        Segreto = 'nessuna';                        Thumbprint = 'quella del certificato (anni)';   CA_propria = 'la scegli tu (qui 90 giorni)';     ADR = 'la policy (qui 30 giorni)' },
  [ordered]@{ Aspetto = 'Rinnovo';                       Segreto = 'riscrivi device e Hub';          Thumbprint = 'thumbprint secondario, poi scambio'; CA_propria = 'nuovo leaf; il cloud non cambia';  ADR = 'il device rifa la richiesta (CSR)' },
  [ordered]@{ Aspetto = 'Nuovo device';                  Segreto = 'crei identita + segreto';        Thumbprint = 'registri ogni impronta';          CA_propria = 'nessuna modifica al cloud';        ADR = 'nessuna modifica al cloud' },
  [ordered]@{ Aspetto = 'Revoca';                        Segreto = 'rigeneri la chiave: tempi non garantiti'; Thumbprint = "disabiliti l'identita";           CA_propria = 'a tuo carico';                     ADR = 'dal namespace: immediata' },
  [ordered]@{ Aspetto = 'Vista di flotta';               Segreto = 'un registry per Hub';            Thumbprint = 'un registry per Hub';             CA_propria = 'un registry per Hub';              ADR = 'un namespace per tutti gli Hub' }
)
($rows | ForEach-Object { [pscustomobject]$_ }) | Format-Table -AutoSize -Wrap | Out-String -Width 220


Aspetto                Segreto                       Thumbprint                         CA_propria                       ADR
-------                -------                       ----------                         ----------                       ---
Cosa conserva il cloud il segreto stesso             solo l'impronta                    il certificato della CA          la CA (nel namespace)
Chi emette             tu (una stringa)              il device (self-signed)            la tua CA, un file sul tuo disco ADR: Issuing CA con chiavi in HSM
Se il registry trapela tutti i device sono clonabili nulla di utilizzabile              nulla di utilizzabile            nulla di utilizzabile
Durata                 nessuna                       quella del certificato (anni)      la scegli tu (qui 90 giorni)     la policy (qui 30 giorni)
Rinnovo                riscrivi device e Hub         thumbprint secondario, poi scambio nuovo leaf; il cloud non cambia  il device rifa la richiesta (CSR)
Nuovo 

**Da dire.** "La differenza non è la crittografia, è **dove sta il segreto** e **chi si prende il ciclo di vita**. Con il segreto
condiviso, chi legge il registry può impersonare tutti. Con il thumbprint no, ma la fiducia è device per device. Con una CA propria
la fiducia scala, ma la CA sei tu. Con ADR la CA è una risorsa e il rinnovo è un'operazione normale del device: lo vediamo nel notebook 06."

Sulla **revoca** c'è una differenza che conta: con il segreto condiviso, anche rigenerando la chiave **non c'è garanzia su quanto tempo ci vuole** perché l'accesso cada; con il certificato la revoca è **immediata** (lo si vede nel notebook 10: il primo messaggio dopo la revoca è già rifiutato).

## 6 · Gli stessi casi sull'Hub
Le prime due celle creano due device identity di prova sull'Hub: una con segreto condiviso, una con thumbprint. Su un Hub
collegato ad ADR (la modalità storica solo-Hub usa un Hub non collegato). Le identità si cancellano alla fine.

In [ ]:
$hub = $hubNames[0]
Grant-HubDataAccess -HubName $hub -ResourceGroup $rg
az iot hub device-identity create --hub-name $hub --resource-group $rg --device-id demo-sas --auth-method shared_private_key --pk $sas.Primary --sk $sas.Secondary --auth-type login -o none
$thumb = $self.Thumbprint
az iot hub device-identity create --hub-name $hub --resource-group $rg --device-id demo-thumb --auth-method x509_thumbprint --primary-thumbprint $thumb --auth-type login -o none
az iot hub device-identity list --hub-name $hub --resource-group $rg --auth-type login --query "[?starts_with(deviceId,'demo-')].{device:deviceId, autenticazione:authentication.type}" -o table

Salto: $azure = $false.

In [ ]:
foreach ($id in 'demo-sas', 'demo-thumb') { az iot hub device-identity delete --hub-name $hubNames[0] --resource-group $rg --device-id $id --auth-type login --only-show-errors }
"Identita di prova eliminate."

Salto: $azure = $false.

## 7 · La vista dell'operation
Il problema da cui siamo partiti: chi lavora in operation, senza un'applicazione dedicata, usa il portale o PowerShell. Senza ADR
**ogni Hub è un inventario**: bisogna interrogarli uno a uno e ricomporre. Con ADR c'è **una sola domanda** al namespace.

In [ ]:
foreach ($h in $hubNames) { Grant-HubDataAccess -HubName $h -ResourceGroup $rg }      # il ruolo sui dati serve su ogni Hub
"--- SENZA ADR: un giro, un Hub alla volta"
$tutti = foreach ($h in $hubNames) {
    az iot hub device-identity list --hub-name $h --resource-group $rg --auth-type login --query "[].{device:deviceId, hub:'$h'}" -o json --only-show-errors | ConvertFrom-Json
}
$tutti | Format-Table | Out-String
"--- CON ADR: una domanda al namespace"
az iot adr ns device list --namespace $adrName --resource-group $rg -o table

Salto: $azure = $false. Senza ADR si scrive questo giro, e va ripetuto per ogni Hub:

foreach ($h in $hubNames) { az iot hub device-identity list --hub-name $h ... }

> **Da fare a mano:** aggiungere un attributo al device (`az iot adr ns device attribute create`) e poi filtrare per quell'attributo, per dare
> alla flotta una descrizione che **segue il device anche se cambia Hub**. Leggi prima `az iot adr ns device attribute create -h`.

> **Proprietà del device.** *Manufacturer*, *model* e *software revision* non sono attributi ma proprietà, e il comando è: `az iot adr ns device update --manufacturer … --model … --software-revision …` (notebook 02c). L'attributo di sopra, invece, descrive la flotta (per esempio `fleet`) e non è un'impostazione che il device legge e applica.

**Prossimo notebook: 07, il rinnovo del certificato con un device vero.**